In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from skimage.measure import shannon_entropy
import tensorflow as tf

In [3]:
dataset_path = r"F:\work\python\image frequency\images"   # original dataset
output_path = r"F:\work\python\image frequency\processed_dataset_fixed"

In [4]:
data = []

for class_name in os.listdir(dataset_path):
    class_folder = os.path.join(dataset_path, class_name)
    
    if os.path.isdir(class_folder):
        for img_name in os.listdir(class_folder):
            img_path = os.path.join(class_folder, img_name)
            
            data.append({
                "path": img_path,
                "class": class_name
            })

df = pd.DataFrame(data)
print("Total images:", len(df))

Total images: 1691


In [5]:
features = []

for row in tqdm(df.itertuples(), total=len(df)):
    
    img = cv2.imread(row.path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (224, 224))

    mean_intensity = np.mean(gray)

    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    entropy = shannon_entropy(gray)

    features.append({
        "path": row.path,
        "class": row._2,
        "intensity": mean_intensity,
        "edge_density": edge_density,
        "entropy": entropy
    })

df = pd.DataFrame(features)

100%|██████████| 1691/1691 [00:04<00:00, 360.54it/s]


In [6]:
def categorize(row):
    if row["edge_density"] < 0.05:
        return "low"
    elif row["edge_density"] < 0.15:
        return "medium"
    else:
        return "high"

df["category"] = df.apply(categorize, axis=1)

print(df["category"].value_counts())

category
medium    1106
low        393
high       192
Name: count, dtype: int64


In [7]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [8]:
for split in ["train", "test"]:
    for cat in ["low", "medium", "high"]:
        for cls in df["class"].unique():
            os.makedirs(os.path.join(output_path, split, cat, cls), exist_ok=True)

In [9]:
def save_images(dataframe, split):
    
    for row in tqdm(dataframe.itertuples(), total=len(dataframe)):
        
        img = cv2.imread(row.path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (224, 224))

        save_dir = os.path.join(
            output_path,
            split,
            row.category,
            row._2   # class label
        )

        filename = os.path.basename(row.path)
        save_path = os.path.join(save_dir, filename)

        cv2.imwrite(save_path, gray)

In [10]:
save_images(train_df, "train")
save_images(test_df, "test")

100%|██████████| 339/339 [00:00<00:00, 1112.09it/s]


In [11]:
def load_dataset(path):
    return tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(224, 224),
        batch_size=32,
        color_mode='grayscale',
        label_mode='categorical'
    )

In [12]:
def preprocess(ds):
    ds = ds.map(lambda x, y: (x/255.0, y))
    ds = ds.map(lambda x, y: (tf.image.grayscale_to_rgb(x), y))
    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [13]:
low_train = preprocess(load_dataset(os.path.join(output_path, "train", "low")))
low_test  = preprocess(load_dataset(os.path.join(output_path, "test", "low")))

Found 311 files belonging to 10 classes.
Found 82 files belonging to 10 classes.


In [14]:
from tensorflow.keras import layers, models

def build_simple_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [15]:
model = build_simple_cnn()

history = model.fit(
    low_train,
    validation_data=low_test,
    epochs=5
)

Epoch 1/5


C:\Users\gopik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 538ms/step - accuracy: 0.6817 - loss: 3.3675 - val_accuracy: 0.8415 - val_loss: 0.9013
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 542ms/step - accuracy: 0.8971 - loss: 0.4773 - val_accuracy: 0.8780 - val_loss: 0.4822
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 485ms/step - accuracy: 0.9228 - loss: 0.2326 - val_accuracy: 0.8780 - val_loss: 0.4168
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 464ms/step - accuracy: 0.9711 - loss: 0.1032 - val_accuracy: 0.9024 - val_loss: 0.4507
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 467ms/step - accuracy: 0.9871 - loss: 0.0590 - val_accuracy: 0.9024 - val_loss: 0.4136
